In [ ]:
# Load Google Drive into Google Colab
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
%cd /content/gdrive/My Drive/PhD Journey/Shared_Task

/content/gdrive/My Drive/PhD Journey/Shared_Task


In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm
!python -m spacy download es_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 71.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 79.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split

In [ ]:
import spacy

nlp_en = spacy.load("en_core_web_sm")
nlp_es = spacy.load("es_core_news_sm")

**English Task**

In [ ]:
train_df_en = pd.read_csv("train_en_2000.csv",sep=";")
train_df_en.head()

,id,context,question,answer
0,1,– To provide a significant proportion of perfo...,What is the consequence of providing a signifi...,aligning management with shareholders' interes...
1,2,The vast majority of the Group's bank borrowin...,Why is there a risk that more than one propert...,The vast majority of the Group's bank borrowin...
2,3,Brazil has significant long-term potential and...,What did Brazil's long-term potential and the ...,the creation of a very valuable asset in Brazi...
3,4,We are not proposing changes to the 2017 Perfo...,What impact did the decision to not propose ch...,our incentive arrangements will be linked only...
4,5,"In addition, future interest charges will also...",What is the reason for the anticipated increas...,the interest rate swap became ineffective at t...


In [ ]:
len(train_df_en)

2000

In [ ]:
train_df_en[["id", "context", "question", "answer"]].isna().sum()

,0
id,0
context,0
question,0
answer,0


In [ ]:
train_df_en.duplicated(subset=["context", "question", "answer"]).sum()

np.int64(0)

In [ ]:
cols = ["context", "question", "answer"]

In [ ]:
# Compute descriptive statistics of word counts for selected text columns.
# Each column is converted to string, split into tokens by whitespace,
# and the number of words per row is summarised using .describe().
train_df_en[cols].apply(lambda x: x.astype(str).str.split().str.len().describe())

,context,question,answer
count,2000.00000,2000.000000,2000.000000
mean,46.09050,14.088500,15.887000
std,21.00926,5.338625,10.161784
min,9.00000,4.000000,1.000000
25%,36.00000,10.000000,9.000000
50%,44.00000,13.000000,14.000000
75%,49.00000,17.000000,21.000000
max,320.00000,49.000000,137.000000


In [ ]:
# ============================================
# Verify Extractive Property (Verbatim Answers)
# ============================================

def check_extractive_property(df):
    total = len(df)
    verbatim_count = 0
    mismatch_examples = []

    for _, row in df.iterrows():
        context = str(row["context"])
        answer = str(row["answer"])

        # Check if answer appears exactly in the context
        if answer in context:
            verbatim_count += 1
        else:
            # Store first few mismatches for inspection
            if len(mismatch_examples) < 5:
                mismatch_examples.append({
                    "context_snippet": context[:200],
                    "answer": answer
                })

    percentage = (verbatim_count / total) * 100

    print(f"Total samples: {total}")
    print(f"Verbatim matches: {verbatim_count}")
    print(f"Percentage extractive: {percentage:.2f}%")

    if mismatch_examples:
        print("\nExamples where answer is NOT verbatim:")
        for ex in mismatch_examples:
            print("\nAnswer:", ex["answer"])
            print("Context snippet:", ex["context_snippet"])

In [ ]:
check_extractive_property(train_df_en)

Total samples: 2000
Verbatim matches: 2000
Percentage extractive: 100.00%


In [36]:
# ============================================
# 1) Regex Patterns (CAUSE / EFFECT / OTHER)
# ============================================

# EFFECT triggers (effects / outcomes / consequences)
EFFECT_PAT = re.compile(
    r"\b("
    r"what did .+ (lead|contribute) to|"
    r"leads? to|"
    r"led to|"
    r"result(ed)? in|"
    r"result of|"
    r"as a result|"
    r"consequence(s)?|"
    r"outcome(s)?|"
    r"effect(s)?|"
    r"impact|"
    r"implication(s)?|"
    r"bring about|"
    r"brought about|"
    r"give rise to|"
    r"what happened (because|as a result|due to)|"
    r"what were the implications|"
    r"what was the impact|"
    r"what are the consequences"
    r")\b",
    re.IGNORECASE,
)

# CAUSE triggers (reasons / drivers / explanations)
CAUSE_PAT = re.compile(
    r"\b("
    r"reason(s)?|"
    r"main reason|"
    r"what (is|was|were) the cause|"
    r"why (is|was|were|did|does|do)|"
    r"why (is|was|were|are|did|does|do|has|have|had|can|could|will|would|should|might|may)\b"
    r"factor(s)?|"
    r"driver(s)?|"
    r"key driver(s)?|"
    r"owed to|"
    r"reasons behind|"
    r"contribut(e|ed|es|ing)|"
    r"drove|"
    r"driven by|"
    r"prompt(ed|s|ing)|"
    r"motiv(at|ated|ates|ating)|"
    r"explain(s|ed)?|"
    r"explanation|"
    r"stem(s)? from|"
    r"account(s)? for|"
    r"what (is|was) behind|"
    r"responsible for|"
    r"due to|"
    r"because|"
    r"caus(e|ed|es|ing)|"
    r"trigger(ed|s|ing)|"
    r"attribut(ed|able) to|"
    r"result(ed)? from|"
    r"foster(s|ed|ing)?|"
    r"what (originated|explains|prompted)|"
    r"what has helped"
    r")\b",
    re.IGNORECASE,
)

OTHER_PAT = re.compile(
    r"\b("
    r"entail(s|ed)?|"
    r"what does .+ entail|"
    r"what it means|"
    r"imply|implies|implied"
    r")\b",
    re.IGNORECASE,
)

# Override: "reason/factors/why ... led to" is CAUSE intent
CAUSE_INTENT_OVERRIDE = re.compile(
    r"\b(why|reason(s)?|factor(s)?|what caused)\b.*\bled to\b",
    re.IGNORECASE,
)
# ============================================
# 2) Labelling Function
# ============================================

def label_question(q: str,lang: str = "en") -> str:
    """
    Label a question as CAUSE / EFFECT / OTHER using regex patterns.

    Rules:
    - If multiple patterns match, return 'AMBIGUOUS'
    - If none match, return 'UNMATCHED'
    """
    q = str(q)

    matches = {
        "CAUSE": bool(CAUSE_PAT.search(q)),
        "EFFECT": bool(EFFECT_PAT.search(q)),
        "OTHER": bool(OTHER_PAT.search(q)),
    }
    hit = [k for k, v in matches.items() if v]

    if len(hit) == 1:
        return hit[0]
    if len(hit) > 1:
        return "AMBIGUOUS"
    return "UNMATCHED"


# ============================================
# 3) Apply to a DataFrame and Summarise
# ============================================

# Example: analyse English training set
df = train_df_en.copy()

# Create a pattern label column
df["q_pattern"] = df["question"].apply(label_question)

# Distribution
counts = df["q_pattern"].value_counts(dropna=False)
pct = (counts / len(df) * 100).round(2)

summary = pd.DataFrame({"count": counts, "percent": pct})
display(summary)

,count,percent
q_pattern,,
CAUSE,1060,53.00
EFFECT,576,28.80
UNMATCHED,275,13.75
AMBIGUOUS,48,2.40
OTHER,41,2.05


In [34]:
#train_df_en
# ============================================
# 4) Inspect Examples
# ============================================

def show_examples(train_df_en, label, n=10):
    """Print a few sample questions for a given label."""
    sample = df[df["q_pattern"] == label].sample(
        n=min(n, (df["q_pattern"] == label).sum()),
        random_state=42
    )
    for i, q in enumerate(sample["question"].tolist(), 1):
        print(f"{i}. {q}")

for label in ["CAUSE", "EFFECT", "OTHER", "AMBIGUOUS", "UNMATCHED"]:
    print("\n" + "="*60)
    print(label)
    print("="*60)
    if (df["q_pattern"] == label).any():
        show_examples(df, label, n=8)
    else:
        print("No samples.")


CAUSE
1. What factor explains that free cash flow is the most appropriate measure of cash flow performance?
2. Why were none of the 2015 LTIP awards that vest subject to deferral?
3. What accounts for the absence of a "minimum order quantity"?
4. What was the reason for the missed opportunities?
5. What does the increase in input cost result from?
6. What caused the Total Marketable Coal Reserves to decrease?
7. What is the reason behind the importance of drawing directors from the widest talent pool?
8. What factor caused the loss of a total of 119 working days?

EFFECT
1. What effect did the scale and mid-year timing of the Skyepharma merger and the shortened nine-month accounting period in 2016 have?
2. What was the outcome of translating the Group's non-Sterling denominated balance sheet with a strong US dollar against Sterling?
3. What were the effects of the Group also performing reverse stress tests to help management understand the full range of potential adverse impacts?
4. W

**Define Causal Markers**

In [ ]:
CAUSAL_MARKERS_EN = {
    "because", "due", "result", "resulting",
    "led", "lead", "cause", "caused",
    "therefore", "hence", "thus"
}

In [ ]:
def count_causal_chains(text, nlp, markers):
    """
    Approximate the number of causal chains in a text.

    Strategy:
    - Count causal discourse markers
    - Count causal verbs
    - Count subordinating conjunctions introducing causal clauses
    """

    doc = nlp(str(text))

    chain_count = 0

    for token in doc:
        # Marker match
        if token.lemma_.lower() in markers:
            chain_count += 1

        # Dependency-based detection (causal subordinate clause)
        if token.dep_ in {"mark", "advcl"} and token.text.lower() in markers:
            chain_count += 1

    return chain_count

In [ ]:
train_df_en["context_chains"] = train_df_en["context"].apply(
    lambda x: count_causal_chains(x, nlp_en, CAUSAL_MARKERS_EN)
)

train_df_en["answer_chains"] = train_df_en["answer"].apply(
    lambda x: count_causal_chains(x, nlp_en, CAUSAL_MARKERS_EN)
)

print(train_df_en[["context_chains", "answer_chains"]].describe())

       context_chains  answer_chains
count     2000.000000     2000.00000
mean         0.984500        0.06400
std          0.733845        0.26446
min          0.000000        0.00000
25%          1.000000        0.00000
50%          1.000000        0.00000
75%          1.000000        0.00000
max          5.000000        2.00000


**Sentence Counting Function**

In [ ]:
def count_sentences(text, nlp):
    """
    Count the number of sentences in a text using spaCy sentence segmentation.
    """
    doc = nlp(str(text))
    return len(list(doc.sents))

In [ ]:
# Count sentences in context
train_df_en["context_sentences"] = train_df_en["context"].apply(
    lambda x: count_sentences(x, nlp_en)
)

# Count sentences in answer
train_df_en["answer_sentences"] = train_df_en["answer"].apply(
    lambda x: count_sentences(x, nlp_en)
)

# Show statistics
print("Context sentence stats:")
print(train_df_en["context_sentences"].describe())

print("\nAnswer sentence stats:")
print(train_df_en["answer_sentences"].describe())

Context sentence stats:
count    2000.000000
mean        1.830500
std         0.848605
min         1.000000
25%         1.000000
50%         2.000000
75%         2.000000
max         8.000000
Name: context_sentences, dtype: float64

Answer sentence stats:
count    2000.000000
mean        1.013000
std         0.129767
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         4.000000
Name: answer_sentences, dtype: float64


In [ ]:
train_df_en

,id,context,question,answer,context_chains,answer_chains,context_sentences,answer_sentences
0,1,– To provide a significant proportion of perfo...,What is the consequence of providing a signifi...,aligning management with shareholders' interes...,1,0,1,1
1,2,The vast majority of the Group's bank borrowin...,Why is there a risk that more than one propert...,The vast majority of the Group's bank borrowin...,1,0,2,1
2,3,Brazil has significant long-term potential and...,What did Brazil's long-term potential and the ...,the creation of a very valuable asset in Brazi...,2,0,4,1
3,4,We are not proposing changes to the 2017 Perfo...,What impact did the decision to not propose ch...,our incentive arrangements will be linked only...,1,0,2,1
4,5,"In addition, future interest charges will also...",What is the reason for the anticipated increas...,the interest rate swap became ineffective at t...,0,0,1,1
...,...,...,...,...,...,...,...,...
1995,1996,■ ■ Barclays UK: Reduction was driven by UK ca...,What caused the Barclays UK's reduction?,UK cards portfolio,1,0,1,1
1996,1997,Inability to obtain necessary water concession...,What causes the inability to obtain necessary ...,government control or private interests,1,0,1,1
1997,1998,Special items amount to €57 million and are ma...,What factors influence the special items that ...,the capital gain on the sale of the interest h...,2,1,1,1
1998,1999,Annualised cost per home account fell by 8% co...,What motivated a decrease in incoming call vol...,investment in our digital platform,1,0,1,1


**Splitting the dataset**

In [ ]:
# 1. Split into 80% Train and 20% Dev
train_df, dev_df = train_test_split(
    train_df_en,
    test_size=0.20, # 20% for evaluation/validation
    random_state=42,
    shuffle=True
)

print(f"Data ready: Train: {len(train_df)} | Dev: {len(dev_df)}")

# 2. Save for your Unsloth pipeline
train_df.to_csv("train_80_en.csv", index=False, sep=";")
dev_df.to_csv("dev_20_en.csv", index=False, sep=";")



Data ready: Train: 1600 | Dev: 400


0

**Spanish Task**

In [48]:
train_df_es = pd.read_csv("train_es_2000.csv",sep=";")
train_df_es.head()

,id,context,question,answer
0,1,Sin duda es importante afrontar los desafíos q...,¿Cuál es la causa de los desafíos que se les p...,la velocidad que ha adquirido el desarrollo y ...
1,2,"Por Áreas, la División de Televisión ha vuelto...",¿Gracias a qué medida la División de Televisió...,"a un modelo reconocible por sus Informativos, ..."
2,3,"Además, después de haber crecido significativa...",¿En qué ha derivado el empeoramiento de las re...,después de haber crecido significativamente en...
3,4,La rentabilidad ha mejorado gracias a un entor...,¿Gracias a qué ha mejorado la rentabilidad?,a un entorno más favorable
4,5,"Es un crecimiento de calidad, ya que está basa...",¿Por qué se dice que es un crecimiento de cali...,está basado en aumentos en los ingresos más co...


In [ ]:
len(train_df_es)

2000

In [ ]:
train_df_es[["id", "context", "question", "answer"]].isna().sum()

,0
id,0
context,0
question,0
answer,0


In [ ]:
train_df_es[cols].apply(lambda x: x.astype(str).str.split().str.len().describe())

,context,question,answer
count,2000.00000,2000.000000,2000.000000
mean,46.94800,16.175500,20.627000
std,24.35627,7.259411,12.067132
min,4.00000,4.000000,2.000000
25%,31.00000,11.000000,12.000000
50%,41.00000,15.000000,18.000000
75%,56.00000,20.000000,27.000000
max,255.00000,83.000000,113.000000


In [ ]:
check_extractive_property(train_df_es)

Total samples: 2000
Verbatim matches: 2000
Percentage extractive: 100.00%


In [38]:
# ============================================
# 1b) Spanish Regex Patterns
# ============================================

EFFECT_PAT_ES = re.compile(
    r"\b("
    r"llev[oó] a|lleva(r)? a|"
    r"result[oó] en|resulta(r)? en|"
    r"como resultado|"
    r"como consecuencia|"
    r"consecuencia(s)?|"
    r"resultado(s)?|"
    r"efecto(s)?|"
    r"impacto|"
    r"implicaci[oó]n(es)?|"
    r"trajo consigo|"
    r"dio lugar a|"
    r"qu[eé] (pas[oó]|ocurri[oó]) (porque|debido a|a causa de)|"
    r"qu[eé] implicaciones (tuvo|tiene)|"
    r"cu[aá]l fue el impacto|"
    r"cu[aá]les (fueron|son) las consecuencias"
    r")\b",
    re.IGNORECASE,
)

CAUSE_PAT_ES = re.compile(
    r"\b("
    r"raz[oó]n(es)?|"
    r"principal raz[oó]n|"
    r"cu[aá]l(es)? (es|fue|fueron|son) (la|el|las|los) causa(s)?|"
    r"por qu[eé]|"
    r"factor(es)?|"
    r"motor(es)?|"
    r"se deb[ei][oó]? a|"
    r"se debe a|"
    r"razones detr[aá]s|"
    r"contribuy[oó]|contribuye|contribuir|"
    r"impuls[oó]|impulsado por|"
    r"provoc[oó]|provocar|"
    r"motiv[oó]|motivado por|"
    r"explic[ao]|explicaci[oó]n|"
    r"surge? de|surg[ií][oó] de|"
    r"da cuenta de|"
    r"qu[eé] (es|fue|estaba) detr[aá]s|"
    r"responsable de|"
    r"debido a|"
    r"porque|"
    r"caus[oó]|causa(r)?|causado por|"
    r"desencaden[oó]|desencadenar|"
    r"atribuido a|atribuible a|"
    r"result[oó] de|resulta de|"
    r"foment[oó]|fomentar|"
    r"qu[eé] (origin[oó]|explica|impuls[oó])|"
    r"qu[eé] ha ayudado"
    r")\b",
    re.IGNORECASE,
)

OTHER_PAT_ES = re.compile(
    r"\b("
    r"implica(r)?|implic[oó]|"
    r"qu[eé] significa|"
    r"qu[eé] quiere decir|"
    r"conlleva(r)?"
    r")\b",
    re.IGNORECASE,
)

CAUSE_INTENT_OVERRIDE_ES = re.compile(
    r"\b(por qu[eé]|raz[oó]n(es)?|factor(es)?|qu[eé] caus[oó])\b.*\b(llev[oó] a|condujo a)\b",
    re.IGNORECASE,
)

# ============================================
# 2b) Updated label_question with lang routing
# ============================================

def label_question(q: str, lang: str = "es") -> str:
    q = str(q)

    if lang == "es":
        effect_pat = EFFECT_PAT_ES
        cause_pat = CAUSE_PAT_ES
        other_pat = OTHER_PAT_ES
        override_pat = CAUSE_INTENT_OVERRIDE_ES
    else:
        effect_pat = EFFECT_PAT
        cause_pat = CAUSE_PAT
        other_pat = OTHER_PAT
        override_pat = CAUSE_INTENT_OVERRIDE

    # Override: "why X led to Y" → CAUSE
    if override_pat.search(q):
        return "CAUSE"

    matches = {
        "CAUSE": bool(cause_pat.search(q)),
        "EFFECT": bool(effect_pat.search(q)),
        "OTHER": bool(other_pat.search(q)),
    }
    hit = [k for k, v in matches.items() if v]

    if len(hit) == 1:
        return hit[0]
    if len(hit) > 1:
        return "AMBIGUOUS"
    return "UNMATCHED"


# ============================================
# 3b) Apply to Spanish DataFrame
# ============================================

df_es = train_df_es.copy()
df_es["q_pattern"] = df_es["question"].apply(lambda q: label_question(q, lang="es"))

counts_es = df_es["q_pattern"].value_counts(dropna=False)
pct_es = (counts_es / len(df_es) * 100).round(2)

summary_es = pd.DataFrame({"count": counts_es, "percent": pct_es})
display(summary_es)

,count,percent
q_pattern,,
UNMATCHED,814,40.70
CAUSE,634,31.70
EFFECT,433,21.65
AMBIGUOUS,64,3.20
OTHER,55,2.75


In [42]:
def show_examples(df, label, n=10, lang="en"):
    """Print sample questions for a given label."""
    subset = df[df["q_pattern"] == label]
    count = len(subset)
    if count == 0:
        print("No samples.")
        return
    sample = subset.sample(n=min(n, count), random_state=42)
    for i, q in enumerate(sample["question"].tolist(), 1):
        print(f"{i}. {q}")
# Spanish
print("\nSPANISH")
for label in ["CAUSE", "EFFECT", "OTHER", "AMBIGUOUS", "UNMATCHED"]:
    print("\n" + "="*60)
    print(label)
    print("="*60)
    show_examples(df_es, label, n=8, lang="es")


SPANISH

CAUSE
1. ¿Por qué en 2015 los flujos netos de efectivo de actividades de explotación alcanzaba los 2.656 millones de euros, en comparación con los 3.714 millones de euros de 2014?
2. La aplicación tiene un carácter social, ¿por qué?
3. ¿Qué motiva que el Grupo aplique una protección especial a todos los migrantes, incluyendo a los trabajadores refugiados con motivo del conflicto en Siria?
4. ¿Qué provocó que los ROF registraran un descenso (-4,6%)?
5. ¿Qué medidas han permitido en la red comercial de BBVA en España fomentar la integración laboral de personas con síndrome de Down?
6. ¿Qué provocó que la deuda neta de las actividades de financiación periódicas descendiera 795 millones de euros?
7. ¿Por qué fue más complicada la situación en el segundo semestre?
8. ¿Por qué ha conseguido convertirse en el grupo líder en comunicación de nuestro país?

EFFECT
1. ¿Qué impacto tiene el nivel actual de los tipos de interés de mercado?
2. ¿Qué implicación tiene ser responsables?
3. ¿Q

In [43]:
# Count sentences in context
train_df_es["context_sentences"] = train_df_es["context"].apply(
    lambda x: count_sentences(x, nlp_es)
)

# Count sentences in answer
train_df_es["answer_sentences"] = train_df_es["answer"].apply(
    lambda x: count_sentences(x, nlp_es)
)

# Show statistics
print("Context sentence stats:")
print(train_df_es["context_sentences"].describe())

print("\nAnswer sentence stats:")
print(train_df_es["answer_sentences"].describe())

Context sentence stats:
count    2000.000000
mean        1.342000
std         0.752542
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         7.000000
Name: context_sentences, dtype: float64

Answer sentence stats:
count    2000.000000
mean        1.033500
std         0.185459
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         3.000000
Name: answer_sentences, dtype: float64


In [44]:
train_df_es

,id,context,question,answer,context_sentences,answer_sentences
0,1,Sin duda es importante afrontar los desafíos q...,¿Cuál es la causa de los desafíos que se les p...,la velocidad que ha adquirido el desarrollo y ...,1,1
1,2,"Por Áreas, la División de Televisión ha vuelto...",¿Gracias a qué medida la División de Televisió...,"a un modelo reconocible por sus Informativos, ...",1,1
2,3,"Además, después de haber crecido significativa...",¿En qué ha derivado el empeoramiento de las re...,después de haber crecido significativamente en...,1,1
3,4,La rentabilidad ha mejorado gracias a un entor...,¿Gracias a qué ha mejorado la rentabilidad?,a un entorno más favorable,1,1
4,5,"Es un crecimiento de calidad, ya que está basa...",¿Por qué se dice que es un crecimiento de cali...,está basado en aumentos en los ingresos más co...,1,1
...,...,...,...,...,...,...
1995,1996,"Carbonell, gracias a su fuerte presencia en Es...",¿A qué se debe que Carbonell se coloque como t...,a su fuerte presencia en España y México (adem...,1,1
1996,1997,Gracias a un modelo económico en el que el rec...,¿A qué se debe que la industria vidriera en Es...,a un modelo económico en el que el reciclaje r...,1,1
1997,1998,"En este sentido, el Banco ha empezado a operar...",¿Qué repercusión ha tenido que el Banco haya e...,ha ampliado su presencia internacional a dieci...,2,1
1998,1999,"Para llevar a cabo esta función, el Grupo DIA ...",¿Qué consecuencia tiene el procedimiento de se...,"están definidos los recursos, responsabilidade...",1,1


**Splitting the dataset**

In [50]:
# 1. Split into 80% Train and 20% Dev
train_df, dev_df = train_test_split(
    train_df_es,
    test_size=0.20, # 20% for evaluation/validation
    random_state=42,
    shuffle=True
)

print(f"Data ready: Train: {len(train_df)} | Dev: {len(dev_df)}")

# 2. Save for your Unsloth pipeline
train_df.to_csv("train_80_es.csv", index=False, sep=";")
dev_df.to_csv("dev_20_es.csv", index=False, sep=";")


Data ready: Train: 1600 | Dev: 400
